# AOS DOF Audit — LUT / Trim / Tweak / Applied (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-14
**Last Modified:** 2026-07-14
**Status:** In Progress
**Keywords:** AOS, MTAOS, ts_ofc, degree of freedom, hexapod, M1M3, M2, LUT, Trim, Tweak, EFD, ConsDB

## Description

Audit the AOS degree-of-freedom chain per image (seq_num) from the EFD, with
ConsDB / Butler cross-checks:

```
optical_state --PID--> Tweak(visitDoF) --accumulate--> Trim(aggregatedDoF)
Trim --(ts_ofc)--> per-component command --+ LUT(el,T)--> Applied
```

Reads `output/<day_obs>/dof_audit.parquet` (from `code/aos_dof_audit.py`, run on
the Summit RSP) and checks:
1. Trim / Tweak time history + accumulation self-consistency (`Trim_i - Trim_{i-1}` vs `Tweak_i`).
2. Hexapod LUT (`compensated - uncompensated`) vs elevation, and command vs applied.
3. OFC PID reasonableness (`wavefrontError` input vs `visitDoF` output).
4. EFD <-> ConsDB consistency (elevation, Zernikes) where they overlap.
5. Mirror-mode forces: MTAOS command vs M1M3 applied active-optic forces.

**Output:** diagnostic figures (in-notebook).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-14 | Aaron Roodman | Initial version — DOF audit diagnostics from dof_audit.parquet |

## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
from pathlib import Path
DAY_OBS = 20260713
PARQUET = Path("output") / str(DAY_OBS) / "dof_audit.parquet"
KEY_NOLL = [4, 5, 6, 7, 8, 11]      # Zernikes to show for the PID / ConsDB checks
OBS_TYPES = ("science",)            # keep these obs_type (None = all); acq are FAM/acquisition
DROP_FAM = True                     # drop FAM-triplet / defocus visits (open loop, |cam z cmd|>500 µm)

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DOF_GROUPS = {"M2 hex": range(0, 5), "Cam hex": range(5, 10),
              "M1M3 bend": range(10, 30), "M2 bend": range(30, 50)}
HEX_AXES = ["x", "y", "z", "u", "v", "w"]


def arr_col(df, col):
    """List-valued column -> 2D array (n_rows, n)."""
    v = df[col].values
    return np.array([np.asarray(x, float) if x is not None else np.full(1, np.nan) for x in v], dtype=object) \
        if any(x is None for x in v) else np.array([np.asarray(x, float) for x in v])

## 3. Load

In [ ]:
df = pd.read_parquet(PARQUET)
print("raw:", len(df), "visits;  obs_type:",
      df.obs_type.value_counts(dropna=False).to_dict() if "obs_type" in df else "n/a")
# keep closed-loop science, drop FAM-triplet defocus visits
if OBS_TYPES and "obs_type" in df:
    df = df[df.obs_type.isin(OBS_TYPES)]
if DROP_FAM and "cam_cmd_z" in df:
    df = df[df.cam_cmd_z.abs() < 500]
df = df.sort_values("seq_num").reset_index(drop=True)
seq = df.seq_num.to_numpy()
ordn = np.arange(len(df))
print(f"{len(df)} closed-loop visits, seq {seq.min()}..{seq.max()}, {df.shape[1]} cols")
if "latency_s" in df:
    print(f"OFC correction latency after exposure (TAI): median {df.latency_s.median():.1f}s "
          f"[{df.latency_s.min():.1f}..{df.latency_s.max():.1f}]  (telemetry matched at exposure time)")
trim = np.column_stack([df[f"trim{i}"] for i in range(50)])       # (n, 50)
tweak = np.column_stack([df[f"tweak{i}"] for i in range(50)])
print("Trim finite:", np.isfinite(trim).all(1).sum(), "| Tweak finite:", np.isfinite(tweak).all(1).sum())

## 4. Trim / Tweak + accumulation check

In [ ]:
# --- Trim & Tweak per DOF group + accumulation self-consistency ---
fig, axes = plt.subplots(len(DOF_GROUPS), 2, figsize=(15, 3.0 * len(DOF_GROUPS)),
                         constrained_layout=True, squeeze=False)
for r, (name, idxs) in enumerate(DOF_GROUPS.items()):
    idxs = list(idxs)
    for i in idxs:
        axes[r, 0].plot(ordn, trim[:, i], lw=0.7, alpha=0.8)
        axes[r, 1].plot(ordn, tweak[:, i], lw=0.7, alpha=0.8)
    axes[r, 0].set_ylabel(f"{name}\nTrim"); axes[r, 1].set_ylabel("Tweak")
    for c in (0, 1):
        axes[r, c].grid(alpha=0.3); axes[r, c].axhline(0, color="k", lw=0.4)
axes[0, 0].set_title("Trim = aggregatedDoF"); axes[0, 1].set_title("Tweak = visitDoF")
axes[-1, 0].set_xlabel("visit ordinal"); axes[-1, 1].set_xlabel("visit ordinal")
fig.suptitle(f"DOF Trim / Tweak per group — {DAY_OBS}", fontsize=13)
plt.show()

# accumulation: Trim_i - Trim_{i-1} should equal Tweak_i (per DOF)
dtrim = np.diff(trim, axis=0)
resid = dtrim - tweak[1:]
acc = np.nanmax(np.abs(resid), axis=1)
fig, ax = plt.subplots(figsize=(14, 3), constrained_layout=True)
ax.plot(ordn[1:], acc, ".-", ms=3, color="crimson")
ax.set_yscale("log"); ax.set_xlabel("visit ordinal")
ax.set_ylabel("max|dTrim - Tweak|"); ax.grid(alpha=0.3)
ax.set_title(f"Trim accumulation check: max over DOF of |Trim_i - Trim_(i-1) - Tweak_i| — {DAY_OBS}")
plt.show()
print(f"accumulation residual: median {np.nanmedian(acc):.2e}, 95pct {np.nanpercentile(acc,95):.2e}")

## 5. Hexapod: LUT vs elevation, command vs applied

In [ ]:
# --- Hexapod: LUT (comp - uncomp) vs elevation, and command vs applied ---
el = df.elevation.to_numpy()
for nm in ("cam", "m2"):
    if f"{nm}_comp_z" not in df:
        continue
    fig, axes = plt.subplots(2, 6, figsize=(20, 6), constrained_layout=True)
    for a, ax in enumerate(HEX_AXES):
        lut = df[f"{nm}_lut_{ax}"].to_numpy()
        # top: LUT vs elevation
        axes[0, a].scatter(el, lut, s=8, alpha=0.5)
        axes[0, a].set_title(f"{ax}: LUT (comp-uncomp)"); axes[0, a].set_xlabel("elevation [deg]")
        axes[0, a].grid(alpha=0.3)
        # bottom: command (MTAOS) vs applied AOS target (uncompensated) residual
        d = df[f"{nm}_cmd_{ax}"].to_numpy() - df[f"{nm}_uncomp_{ax}"].to_numpy()
        axes[1, a].plot(ordn, d, ".-", ms=2, lw=0.4)
        axes[1, a].set_title(f"{ax}: cmd - uncomp"); axes[1, a].set_xlabel("visit ordinal")
        axes[1, a].axhline(0, color="k", lw=0.4); axes[1, a].grid(alpha=0.3)
    fig.suptitle(f"{nm.upper()} hexapod — LUT vs elevation (top) | MTAOS cmd - hexapod uncompensated (bottom) — {DAY_OBS}",
                 fontsize=12)
    plt.show()

## 6. OFC PID: wavefront-error input vs DOF Tweak

In [ ]:
# --- OFC PID: wavefrontError (input) vs visitDoF/Tweak (output) ---
if "wfe_values" in df and df["wfe_values"].notna().any():
    wv = np.array([np.asarray(x, float) if x is not None else np.full(50, np.nan)
                   for x in df["wfe_values"].values])
    wn = np.array([np.asarray(x, float) if x is not None else np.full(50, np.nan)
                   for x in df["wfe_noll"].values])
    # column index of each Noll in the (per-row constant) noll list
    noll0 = wn[np.isfinite(wn).all(1)][0].astype(int) if np.isfinite(wn).any() else np.arange(4, 4 + wv.shape[1])
    jpos = {int(j): k for k, j in enumerate(noll0)}
    fig, axes = plt.subplots(len(KEY_NOLL), 1, figsize=(14, 2.2 * len(KEY_NOLL)),
                             constrained_layout=True, squeeze=False)
    for r, j in enumerate(KEY_NOLL):
        ax = axes[r, 0]
        if j in jpos:
            ax.plot(ordn, wv[:, jpos[j]], ".-", ms=2, lw=0.5, color="steelblue", label=f"wfe Z{j} (input)")
        ax.set_ylabel(f"Z{j}"); ax.grid(alpha=0.3); ax.axhline(0, color="k", lw=0.4)
        ax.legend(fontsize=7, loc="upper left")
    axes[-1, 0].set_xlabel("visit ordinal")
    fig.suptitle(f"OFC input wavefront error (key Zernikes) — {DAY_OBS}", fontsize=12)
    plt.show()
    # input magnitude vs output magnitude
    fig, ax = plt.subplots(figsize=(14, 3), constrained_layout=True)
    ax.plot(ordn, np.sqrt(np.nansum(wv ** 2, 1)), ".-", ms=2, lw=0.5, color="steelblue", label="|wfe| (input)")
    ax.plot(ordn, np.sqrt(np.nansum(tweak ** 2, 1)), ".-", ms=2, lw=0.5, color="crimson", label="|Tweak| (output)")
    ax.set_xlabel("visit ordinal"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title(f"OFC PID magnitudes: wavefront-error input vs DOF Tweak output — {DAY_OBS}")
    plt.show()
else:
    print("no wfe_values — skipping PID section")

## 7. EFD <-> ConsDB consistency

In [ ]:
# --- EFD <-> ConsDB consistency (where overlapping) ---
if "cdb_elevation" in df:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    axes[0].scatter(df.elevation, df.cdb_elevation, s=10, alpha=0.6)
    lim = [np.nanmin([df.elevation.min(), df.cdb_elevation.min()]),
           np.nanmax([df.elevation.max(), df.cdb_elevation.max()])]
    axes[0].plot(lim, lim, "k:"); axes[0].set_xlabel("EFD MTMount elevation")
    axes[0].set_ylabel("ConsDB altitude"); axes[0].set_title("elevation: EFD vs ConsDB")
    axes[0].grid(alpha=0.3)
    axes[1].plot(ordn, df.elevation - df.cdb_elevation, ".-", ms=2, lw=0.4)
    axes[1].axhline(0, color="k", lw=0.4); axes[1].set_xlabel("visit ordinal")
    axes[1].set_ylabel("EFD - ConsDB [deg]"); axes[1].set_title("elevation difference"); axes[1].grid(alpha=0.3)
    fig.suptitle(f"EFD <-> ConsDB elevation consistency — {DAY_OBS}", fontsize=12)
    plt.show()
else:
    print("no ConsDB columns (cdb_elevation) — rerun the extractor with ConsDB enabled")

## 8. Mirror-mode forces: command vs applied

In [ ]:
# --- Mirror-mode forces: MTAOS command vs M1M3 applied active-optic forces ---
if "m1m3_cmd_zForces" in df and df["m1m3_cmd_zForces"].notna().any():
    cmd = np.array([np.asarray(x, float) if x is not None else None for x in df["m1m3_cmd_zForces"].values],
                   dtype=object)
    app = np.array([np.asarray(x, float) if x is not None else None for x in df["m1m3_applied_zForces"].values],
                   dtype=object)
    dmax = np.array([np.nanmax(np.abs(c - a)) if (c is not None and a is not None and len(c) == len(a))
                     else np.nan for c, a in zip(cmd, app)])
    fig, ax = plt.subplots(figsize=(14, 3), constrained_layout=True)
    ax.plot(ordn, dmax, ".-", ms=3, color="darkgreen")
    ax.set_xlabel("visit ordinal"); ax.set_ylabel("max|cmd - applied| [N]"); ax.grid(alpha=0.3)
    ax.set_title(f"M1M3 active-optic forces: MTAOS command vs applied (max over 156 actuators) — {DAY_OBS}")
    plt.show()
    # one representative visit: cmd vs applied per actuator
    k = int(np.nanargmax([len(c) if c is not None else 0 for c in cmd]))
    if cmd[k] is not None and app[k] is not None:
        fig, ax = plt.subplots(figsize=(14, 3), constrained_layout=True)
        ax.plot(cmd[k], label="command"); ax.plot(app[k], label="applied", alpha=0.7)
        ax.set_xlabel("M1M3 actuator"); ax.set_ylabel("zForce [N]"); ax.legend(); ax.grid(alpha=0.3)
        ax.set_title(f"M1M3 forces cmd vs applied — visit {int(df.visit_id.iloc[k])}")
        plt.show()
else:
    print("no m1m3 force columns — skipping mirror-force section")